# UC09 — Inteligencia sobre Quejas de Clientes

Clasificación NLP y análisis de sentimiento sobre reclamaciones.
**Valor de negocio**: Detectar problemas emergentes y riesgo regulatorio de forma proactiva.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS QUEJAS_CLIENTES_DB;
USE SCHEMA QUEJAS_CLIENTES_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Quejas Sintéticas (500)

In [ ]:
CREATE OR REPLACE TABLE quejas AS
SELECT 'QUE' || LPAD(SEQ4(),5,'0') AS queja_id, DATEADD(day,-UNIFORM(1,180,RANDOM()),CURRENT_DATE()) AS fecha,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Email' WHEN 1 THEN 'Telefono' WHEN 2 THEN 'Web' ELSE 'RRSS' END AS canal,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Auto' WHEN 1 THEN 'Hogar' WHEN 2 THEN 'Vida' ELSE 'Salud' END AS producto,
    CASE MOD(SEQ4(),8)
        WHEN 0 THEN 'Llevo 3 meses esperando el pago de mi siniestro. Ya he llamado 5 veces y nadie me da solución.'
        WHEN 1 THEN 'La atención telefónica es pésima. Me han tenido 40 minutos en espera y me colgaron.'
        WHEN 2 THEN 'Me han denegado la cobertura del cristal cuando mi póliza dice claramente que está incluido.'
        WHEN 3 THEN 'La subida de prima del 25% es abusiva. No he tenido ningún siniestro en 5 años.'
        WHEN 4 THEN 'El perito vino a mi casa y fue muy amable y profesional. Resolvieron todo en una semana.'
        WHEN 5 THEN 'La web no funciona y el chat no responde. Necesito modificar mi póliza urgentemente.'
        WHEN 6 THEN 'El taller asignado hizo un trabajo horrible. El coche tiene más daños que antes. Pondré denuncia ante la DGSFP.'
        ELSE 'Proceso de siniestro lento pero al final lo resolvieron. Podrían mejorar los tiempos.'
    END AS texto_queja,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.3 THEN 'Escalada' WHEN UNIFORM(0,1,RANDOM())<0.6 THEN 'Abierta' ELSE 'Resuelta' END AS estado
FROM TABLE(GENERATOR(ROWCOUNT=>500));

---

## Paso 3: Análisis de Sentimiento con Cortex

In [ ]:
CREATE OR REPLACE TABLE quejas_sentimiento AS
SELECT q.*,
    SNOWFLAKE.CORTEX.SENTIMENT(texto_queja) AS sentimiento_score,
    CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_queja)<-0.5 THEN 'Muy Negativo'
         WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_queja)<0 THEN 'Negativo'
         WHEN SNOWFLAKE.CORTEX.SENTIMENT(texto_queja)<0.3 THEN 'Neutro'
         ELSE 'Positivo' END AS sentimiento_label
FROM quejas q;

---

## Paso 4: Clasificar Categorías con Cortex

In [ ]:
CREATE OR REPLACE TABLE quejas_clasificadas AS
SELECT qs.*,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Clasifica esta queja en exactamente UNA categoría. Responde SOLO la categoría: Retraso_Pago, Atencion_Cliente, Cobertura, Precio, Proceso_Siniestro, Calidad_Servicio. Queja: ' || texto_queja
    ) AS categoria
FROM quejas_sentimiento qs;

---

## Paso 5: Detectar Riesgo Regulatorio

In [ ]:
CREATE OR REPLACE TABLE quejas_riesgo AS
SELECT *,
    CASE WHEN LOWER(texto_queja) LIKE '%dgsfp%' OR LOWER(texto_queja) LIKE '%defensor%'
         OR LOWER(texto_queja) LIKE '%denuncia%' OR LOWER(texto_queja) LIKE '%abogado%'
    THEN 1 ELSE 0 END AS riesgo_regulatorio
FROM quejas_clasificadas;

---

## Paso 6: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Inteligencia sobre Quejas de Clientes')
df = session.table('quejas_riesgo').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Total Quejas', len(df))
with c2: st.metric('Sentimiento Medio', f"{df['SENTIMIENTO_SCORE'].mean():.2f}")
with c3: st.metric('Riesgo Regulatorio', df['RIESGO_REGULATORIO'].sum())
with c4: st.metric('Escaladas', len(df[df['ESTADO']=='Escalada']))

st.markdown('---')
st.subheader('Sentimiento por Categoría')
cat = df.groupby('CATEGORIA')['SENTIMIENTO_SCORE'].mean().sort_values()
st.bar_chart(pd.DataFrame({'Sentimiento': cat.values}, index=cat.index))

st.markdown('---')
st.subheader('Quejas con Riesgo Regulatorio')
rr = df[df['RIESGO_REGULATORIO']==1][['QUEJA_ID','FECHA','TEXTO_QUEJA','SENTIMIENTO_LABEL','CATEGORIA']]
st.dataframe(rr, use_container_width=True)

st.markdown('---')
prod = st.selectbox('Producto', ['Todos']+list(df['PRODUCTO'].unique()))
fdf = df if prod=='Todos' else df[df['PRODUCTO']==prod]
st.dataframe(fdf[['QUEJA_ID','FECHA','CANAL','PRODUCTO','SENTIMIENTO_LABEL','CATEGORIA','ESTADO']].head(100), use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS QUEJAS_CLIENTES_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;